# EXP07: 点差模型 + V3 Regime + 连续亏损减仓

**合并**: NB18 + NB19 + NB16

1. 点差模型对比 (fixed/atr_scaled/session_aware)
2. V3 ATR Regime 带成本验证 + K-Fold
3. 连续同向亏损后减仓模拟

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS, V3_REGIME
from backtest.stats import print_ranked
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
}
print('Data loaded')

## Part 1: 点差模型对比

In [ ]:
spread_results = []
for model in ["fixed", "atr_scaled", "session_aware"]:
    for base in [0.30, 0.50]:
        kw = {**LIVE_KWARGS, "spread_model": model, "spread_base": base, "spread_cost": base}
        r = run_variant(data, f"{model}_${base}", **kw)
        r['spread_model'] = model
        r['spread_base'] = base
        spread_results.append(r)

print_ranked(spread_results)

## Part 2: V3 ATR Regime 验证

In [ ]:
regime_results = []
for sc in [0.0, 0.30, 0.50]:
    r = run_variant(data, f"Fixed_sp${sc}", **{**LIVE_KWARGS, "spread_cost": sc})
    r['config'] = 'Fixed'
    r['spread'] = sc
    regime_results.append(r)

for sc in [0.0, 0.30, 0.50]:
    r = run_variant(data, f"V3_sp${sc}",
        **{**LIVE_KWARGS, "regime_config": V3_REGIME, "spread_cost": sc})
    r['config'] = 'V3_Regime'
    r['spread'] = sc
    regime_results.append(r)

print_ranked(regime_results)

In [ ]:
for sc in [0.0, 0.30, 0.50]:
    fixed = [r for r in regime_results if r['config'] == 'Fixed' and r['spread'] == sc][0]
    v3 = [r for r in regime_results if r['config'] == 'V3_Regime' and r['spread'] == sc][0]
    print(f"spread=${sc}: Fixed Sharpe={fixed['sharpe']:.2f}, V3 Sharpe={v3['sharpe']:.2f}, "
          f"Delta={v3['sharpe']-fixed['sharpe']:+.2f}")

In [ ]:
print("\n=== V3 Regime K-Fold ($0.50 spread) ===")
v3_folds = run_kfold(data, {**LIVE_KWARGS, "regime_config": V3_REGIME, "spread_cost": 0.50}, label_prefix="V3_")
base_folds = run_kfold(data, {**LIVE_KWARGS, "spread_cost": 0.50}, label_prefix="Base_")

for b, v in zip(base_folds, v3_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  V3={v['sharpe']:.2f}  D={v['sharpe']-b['sharpe']:+.2f}")

## Part 3: 连续同向亏损后减仓模拟

In [ ]:
bl = run_variant(data, "BL", **{**LIVE_KWARGS, "spread_cost": 0.50})
bl_trades = bl['_trades']

def simulate_consecutive_skip(trades, n_threshold):
    consec_buy_loss = 0
    consec_sell_loss = 0
    kept_trades = []
    skipped = 0
    skipped_pnl = 0
    for t in trades:
        should_skip = False
        if t.direction == 'BUY' and consec_buy_loss >= n_threshold:
            should_skip = True
        elif t.direction == 'SELL' and consec_sell_loss >= n_threshold:
            should_skip = True
        if should_skip:
            skipped += 1
            skipped_pnl += t.pnl
            if t.direction == 'BUY': consec_buy_loss = 0
            else: consec_sell_loss = 0
        else:
            kept_trades.append(t)
            if t.pnl < 0:
                if t.direction == 'BUY': consec_buy_loss += 1; consec_sell_loss = 0
                else: consec_sell_loss += 1; consec_buy_loss = 0
            else:
                if t.direction == 'BUY': consec_buy_loss = 0
                else: consec_sell_loss = 0
    return {'threshold': n_threshold, 'kept': len(kept_trades), 'skipped': skipped,
            'kept_pnl': sum(t.pnl for t in kept_trades), 'skipped_pnl': skipped_pnl}

print(f"Baseline: {len(bl_trades)} trades, PnL=${sum(t.pnl for t in bl_trades):.0f}\n")
for n in [3, 4, 5, 6, 7]:
    r = simulate_consecutive_skip(bl_trades, n)
    print(f"N={n}: kept={r['kept']}, skipped={r['skipped']}, "
          f"kept_PnL=${r['kept_pnl']:.0f}, skipped_PnL=${r['skipped_pnl']:.0f} "
          f"({'SAVED' if r['skipped_pnl'] < 0 else 'LOST'} ${abs(r['skipped_pnl']):.0f})")

In [ ]:
for direction in ['BUY', 'SELL']:
    dir_trades = [t for t in bl_trades if t.direction == direction]
    streaks = []
    current_streak = 0
    for t in dir_trades:
        if t.pnl < 0:
            current_streak += 1
        else:
            if current_streak > 0: streaks.append(current_streak)
            current_streak = 0
    if current_streak > 0: streaks.append(current_streak)
    print(f"{direction}: {len(dir_trades)} trades, max streak={max(streaks) if streaks else 0}, "
          f"avg={np.mean(streaks):.1f}, streaks>=5: {sum(1 for s in streaks if s >= 5)}")